<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/scikit-learn/Heart_Disease_Risk_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ushbu sahifada be'morning yurak kasalligi havfi bor yoki yo'qligini bashorat qiluvchi kichik loyiha qilamiz . Dataset AI yordamida yaratiladi.**

- 1 → Yurak kasalligi xavfi bor , 0 → Xavf yo'q

In [1]:
import numpy as np
import pandas as pd
import sklearn

In [2]:
df = pd.DataFrame({
    "age": [29,45,54,38,62,41,57,33,49,60,
            36,52,47,31,58,44,55,39,63,42],

    "cholesterol": [180,220,250,195,270,210,260,185,230,280,
                    200,245,225,190,255,215,240,205,275,212],

    "blood_pressure": [120,140,150,130,160,135,155,125,145,165,
                        132,148,142,128,152,138,149,134,162,136],

    "bmi": [22.5,27.8,30.2,24.1,31.5,26.3,29.8,23.0,28.4,32.1,
            25.2,29.5,27.1,23.8,30.0,26.8,28.9,25.9,31.2,26.5],

    "smoker": [
        "No","Yes","Yes","No","Yes",
        "No","Yes","No","Yes","Yes",
        "No","Yes","No","No","Yes",
        "No","Yes","No","Yes","No"
    ],

    "exercise": [
        "High","Low","Low","Medium","Low",
        "Medium","Low","High","Low","Low",
        "Medium","Low","Medium","High","Low",
        "Medium","Low","Medium","Low","High"
    ],

    "heart_risk": [
        0,1,1,0,1,0,1,0,1,1,
        0,1,1,0,1,0,1,0,1,0
    ]
})
df.head()

,age,cholesterol,blood_pressure,bmi,smoker,exercise,heart_risk
0,29,180,120,22.5,No,High,0
1,45,220,140,27.8,Yes,Low,1
2,54,250,150,30.2,Yes,Low,1
3,38,195,130,24.1,No,Medium,0
4,62,270,160,31.5,Yes,Low,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             20 non-null     int64  
 1   cholesterol     20 non-null     int64  
 2   blood_pressure  20 non-null     int64  
 3   bmi             20 non-null     float64
 4   smoker          20 non-null     object 
 5   exercise        20 non-null     object 
 6   heart_risk      20 non-null     int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 1.2+ KB


In [4]:
# X va y larni ajratish
X = df.drop('heart_risk', axis=1)
y = df['heart_risk']

# sonli va matnli ustunlar nomlarini olish
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=np.object_).columns.tolist()

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# sonli ustunlar uchun pipeline yaratish
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# matnli ustunlar uchun pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])

# transformer yaratish
transformer = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# umumiy pipeline yaratish
full_pipeline = Pipeline([
    ('preprocessing', transformer),
    ('model', LogisticRegression())
])

In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# tekshirmoqchi bo'gan parametrlarni yozish
param_gird = {
    "model__C":[0.1, 1, 10],
    "model__solver":['liblinear', 'lbfgs'],
    'model__max_iter':[100, 300]
}

# balansni saqlagan holda GridSearchCV qilish
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    full_pipeline,
    param_grid=param_gird,
    cv=cv,
    scoring='recall', # yurak kasalligini o'tkizib yubormaslik uchun
    n_jobs=-1
)

# o'qitish
grid_search.fit(X, y)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['age',
                                                                          'cholesterol',
                                                                          'blood_pressure',
                                                                          'bmi']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder())]),
                                                                         ['smoker',
                                                                          'exercise'])])),
                                       ('model', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'model__C': [0.1, 1, 10],
                         'model__max_iter': [100, 300],
                         'model__solver': ['liblinear', 'lbfgs']},
             scoring='recall')

In [7]:
print("Best params:", grid_search.best_params_)
print("Best recall:", grid_search.best_score_)

Best params: {'model__C': 0.1, 'model__max_iter': 100, 'model__solver': 'liblinear'}
Best recall: 0.9


- GridSearchCV biz bergan parametrlar bo'yicha 12 ta kombinatsiyani 5 ta foldli Cross-validation orqali tekshirgan va jami 60 ta model o'qitgan . Ana shundagi natijalar ichidagi eng yuqori o'rtacha Recall natijasini qayd etgan hyperparametrlar : C = 0.1 , solver = liblinear , max_iter = 100 lar bo'lgan . Hamda uning o'rtacha Cross-validation Recalli - 90% bo'lgan  